# LLM-as-a-Judge, AlpacaEval 2.0

Единый ноутбук: отбор задач, генерация ответов/вариантов ответов локальными моделями и прогон судей.
Рассчитан на Kaggle (GPU), **устойчив к обрыву сессии**: все результаты записываются в csv в папку `data/` рядом с ноутбуком после каждого батча, любая ячейка запуска пропускает уже сделанное.

**Пайплайн** (модели работают по очереди, после того, как одна завершает работу, происходит выгрузка из cuda):
1. `Stage 1` - оцениваемые модели (SmolLM2, Gemma-2) отвечают на свои задачи (вариант **A**).
2. `Stage 2` - Phi-3.5-mini строит варианты **B, C, D, E, P** из ответа A.
3. `Stage 3` - судьи (Qwen, Mistral-Nemo, Llama) выносят вердикты по запланированным парам.

**Как продолжить после остановки:** откройте ноутбук, выполните ячейки с настройками/функциями и `Init`, затем снова `Stage 1 → 2 → 3` (или `run_all()`). Если файлов `data/*.csv` в папке ноутбука нет, а результаты прошлого запуска подключены как Kaggle Input - они будут восстановлены автоматически.


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece huggingface_hub

In [ ]:
import os, gc, re, json, time, shutil, hashlib, math, warnings
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

warnings.filterwarnings("ignore", category=FutureWarning)
_T0 = time.time()

# ============================== КОНФИГУРАЦИЯ ==============================
SEED = 42
N_TASKS =  200 #250                  # 200-300
ACTIVE_TASK_LIMIT = 10 #None       # None = все задачи; число K = тест на первых K задачах

# Файлы создаются\читаются в директории ноутбука
DATA_DIR = Path.cwd() / "data"
TASKS_CSV       = DATA_DIR / "tasks.csv"
VARIANTS_CSV    = DATA_DIR / "variants.csv"
COMPARISONS_CSV = DATA_DIR / "comparisons.csv"
JUDGMENTS_CSV   = DATA_DIR / "judgments.csv"
ATTEMPTS_CSV    = DATA_DIR / "answer_attempts.csv"   # журнал попыток генерации ответов A
CLAIMS_CSV      = DATA_DIR / "claims.csv"            # атомарные утверждения ответов A (для D и E)
VATTEMPTS_CSV   = DATA_DIR / "variant_attempts.csv"  # журнал попыток генерации вариантов B-P

# Модели
EVALUATED_MODELS = [                       # каждая отвечает на все задачи, создается вариант A для каждой задачи моделью
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "google/gemma-2-9b-it",
]
SOURCE_TAGS = {                            # короткие метки в variant_id/comparison_id 
    "HuggingFaceTB/SmolLM2-1.7B-Instruct": "smol",
    "google/gemma-2-9b-it": "gemma",
}
REWRITER_MODEL = "microsoft/Phi-3.5-mini-instruct"    # вариант P
GENERATOR_MODEL = REWRITER_MODEL                      # варианты B, C, D, E
JUDGE_MODELS = {                                      # метка (в comparisons.csv)
    "judge_qwen":    "Qwen/Qwen2.5-7B-Instruct",
    "judge_mistral": "mistralai/Mistral-Nemo-Instruct-2407",
    "judge_llama":   "meta-llama/Llama-3.2-3B-Instruct",
}
JUDGES = list(JUDGE_MODELS)
PRIVATE_MODELS = {"meta-llama/Llama-3.2-3B-Instruct", "google/gemma-2-9b-it"}  # нужен HF_TOKEN

# Уровни длины
EXPAND_LEVELS   = [1.25, 2.00]       # B и C   [1.25, 1.50, 2.00]    
COMPRESS_LEVELS = [0.75, 0.50]             # D
LOSSY_LEVELS    = [0.75]                   # E
REPEATS = 2                                # repeat_index 0/1: позиции left/right меняются местами

# Генерация
USE_4BIT = True
BATCH_GEN = 4                              # батч при генерации ответов и вариантов
BATCH_JUDGE = 8                            # батч при судействе
A_MAX_NEW_TOKENS = 700

# Контроль длины ответа A оцениваемой модели (перегенерация)
A_MIN_WORDS, A_MAX_WORDS = 100, 450        # допустимая длина ответа: от неё зависит возможность сжатия до 50% и расширения до 200%
A_MAX_REGEN = 3                            # максимум перегенераций одного ответа
A_TARGET_WORDS = 275                       # ориентир длины в подсказке при перегенерации

# Контроль качества вариантов B-P (длина, утверждения, очистка вывода)
LEN_TOL = 0.10                             # допустимое отклонение длины варианта от цели (B, C, D, P)
VARIANT_MAX_TRIES = 2                      # всего попыток на вариант (1 + одна доработка с обратной связью)
RETRY_SCORE = 0.10                         # штраф <= порога - вариантдостаточно хороший, доработка не нужна
USABLE_SCORE = 0.25                        # к судьям попадают только варианты со штрафом <= порога (остальное-отбраковка)
PAIR_LEN_TOL = 0.15                        # эксперимент 1: B и C допускаются к судейству, если их длины различаются не более чем на 15%
JUDGE_ONLY_USABLE = True                   #судим только пригодные варианты 
BATCH_VARIANTS = 12                        # батч при генерации B-P (fp16, две T4)
QUANTIZE_MODELS = {"google/gemma-2-9b-it", "mistralai/Mistral-Nemo-Instruct-2407"}   # 4бит там, где fp16 не помещается
E_REMOVE_FRAC = 0.30                       # доля атомарных утверждений, удаляемых в E
E_LEN_BAND = (0.50, 0.95)                  # допустимая длина E относительно A
D_MIN_COVERAGE = {0.75: 0.90, 0.50: 0.80}  # доля утверждений A, которые должны остаться в D (по уровню сжатия)
E_MIN_KEPT, E_MAX_REMOVED = 0.90, 0.20     # E: оставшиеся утверждения присутствуют, удалённые - отсутствуют
MIN_CLAIMS = 5                             # меньше утверждений - извлечение повторяется; <3 - проверка покрытия пропускается
CLAIMS_MAX_NEW_TOKENS = 700
BATCH_SCORE = 16                           # батч при оценке по логитам (покрытие утверждений, судьи)
JUDGE_MIN_MASS = 0.20                      # если вероятность токенов A/B меньше -вердикт invalid
TIME_BUDGET_HOURS = 11.0                   # мягкая остановка до лимита сессии Kaggle (ч)

# Критерии отбора (по ответу AlpacaEval)
MIN_WORDS, MAX_WORDS = 100, 450
MIN_SENTENCES = 6
MAX_CODE_FRACTION = 0.30
MIN_ASCII_RATIO = 0.95

TASK_COLS = ["task_id", "source_benchmark", "benchmark_id", "instruction",
             "answers", "gold_answer", "category", "selection_reason"]
VARIANT_COLS = ["variant_id", "task_id", "variant_type", "text", "target_length_ratio",
                "actual_length_ratio", "generator_model", "prompt"]
COMPARISON_COLS = ["comparison_id", "task_id", "experiment_type", "variant_left_id",
                   "variant_right_id", "judge_id", "repeat_index", "random_seed"]
JUDGMENT_COLS = ["judgment_id", "comparison_id", "winner", "raw_response",
                 "latency_ms", "input_tokens", "output_tokens"]
ATTEMPT_COLS = ["variant_id", "attempt", "n_words", "n_tokens", "truncated", "status", "kept", "hint"]
CLAIM_COLS = ["a_variant_id", "claims_json", "removed_json"]
VATTEMPT_COLS = ["variant_id", "attempt", "n_words", "target_words", "rel", "truncated", "preamble", "coverage",
                 "removed_present", "ok", "score", "kept", "feedback"]

DERIVED_PLAN = ([("B", r) for r in EXPAND_LEVELS] + [("C", r) for r in EXPAND_LEVELS]
                + [("D", r) for r in COMPRESS_LEVELS] + [("E", r) for r in LOSSY_LEVELS]
                + [("P", 1.0)])

OUTPUT_RULES = ("Output only the text of the answer itself: no introduction, no notes, no word count, "
                "no quotation marks around it. Keep the original formatting style (paragraphs, lists, markdown).")

PROMPTS = {   # {words}, {lo}, {hi} - целевая длина и допустимый диапазон; {claims_block}/{removed_block} - для D/E
    "B": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nRewrite the answer so that it is {words} words long "
          "(acceptable range: {lo}-{hi} words). Keep every original claim and add ONLY new, relevant, factually "
          "correct claims (extra facts, examples, caveats). " + OUTPUT_RULES),
    "C": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nRewrite the answer so that it is {words} words long "
          "(acceptable range: {lo}-{hi} words) WITHOUT adding any new fact, claim, example or caveat: only paraphrase, "
          "restate, add explanatory connectives and structure. Every original claim must stay. " + OUTPUT_RULES),
    "D": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nCondense the answer to {words} words "
          "(acceptable range: {lo}-{hi} words) while keeping ALL its essential claims.{claims_block}\n"
          "Remove only redundancy, repetition and filler; the text must stay fluent. " + OUTPUT_RULES),
    "E": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nDelete from the answer the points listed below, so "
          "that they do not appear in the text in any form.\n{removed_block}\nKeep all other content and its wording "
          "as close to the original as possible. Do not add anything new. " + OUTPUT_RULES),
    "P": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nRewrite the answer in different wording and style. "
          "Keep exactly the same claims, the same formatting and the same length ({words} words, acceptable range: "
          "{lo}-{hi} words). " + OUTPUT_RULES),
}

CLAIMS_PROMPT = ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nList every distinct atomic claim made in the answer "
                 "(facts, steps, recommendations, caveats), one per line, numbered '1.', '2.', ... Each claim must be a "
                 "short self-contained sentence (at most 25 words) and must not repeat another claim. "
                 "Output only the numbered list.")

COVER_PROMPT = ("Text:\n{text}\n\nClaim: {claim}\n\nDoes the text state or clearly imply this claim? "
                "Answer with a single word: Yes or No.")

JUDGE_PROMPT = ("You are an impartial judge. Below are an instruction and two responses to it. "
                "Decide which response is better. Do not let the order of the responses influence "
                "your decision.\n\n[Instruction]\n{instruction}\n\n[Response A]\n{left}\n\n"
                "[Response B]\n{right}\n\nWhich response is better? "
                "Answer with a single letter: A or B.")


In [ ]:
# ============================== HF TOKEN ==============================
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    print("ВНИМАНИЕ: HF_TOKEN не найден - gated-модели (Llama, Gemma) не загрузятся. "
          "Добавьте токен в Kaggle Secrets.")

quantization_config = None
if torch.cuda.is_available() and USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

_loaded_models = {}


def get_hf_auth_kwargs(model_id):
    return {"token": HF_TOKEN} if (model_id in PRIVATE_MODELS and HF_TOKEN) else {}


def gpu_mem():
    if not torch.cuda.is_available():
        return "cpu"
    return ", ".join(f"cuda:{i} {torch.cuda.memory_allocated(i)/2**30:.1f} GiB"
                     for i in range(torch.cuda.device_count()))


def load_local_model(model_id):
    if model_id in _loaded_models:
        return _loaded_models[model_id]
    auth = get_hf_auth_kwargs(model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left", **auth)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = {"device_map": "auto",
              "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32, **auth}
    quantized = quantization_config is not None and model_id in QUANTIZE_MODELS   # 4-бит только для крупных моделей
    if quantized:
        kwargs["quantization_config"] = quantization_config
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    _loaded_models[model_id] = (tokenizer, model)
    print(f"[load] {model_id} ({'4-bit' if quantized else 'fp16'}) | GPU: {gpu_mem()}")
    return _loaded_models[model_id]


def unload_model(model_id):
    """Полностью освобождает CUDA под следующую модель."""
    entry = _loaded_models.pop(model_id, None)
    if entry is not None:
        tokenizer, model = entry
        del entry, tokenizer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print(f"[unload] {model_id} | GPU: {gpu_mem()}")


@torch.inference_mode()
def generate_batch(model_id, prompts, max_new_tokens, seed=None, **gen_kwargs):
    """Жадная генерация батча. Возвращает список dict(text, input_tokens, output_tokens, latency_ms).
    latency_ms - время батча/размер батча."""
    tokenizer, model = load_local_model(model_id)
    texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}],
                                           tokenize=False, add_generation_prompt=True)
             for p in prompts]
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    add_special_tokens=False).to(model.device)   # BOS уже есть в chat-шаблоне
    t0 = time.perf_counter()
    if seed is not None:
        torch.manual_seed(seed)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id,
                         **{"do_sample": False, **gen_kwargs})
    elapsed_ms = (time.perf_counter() - t0) * 1000
    in_len = enc["input_ids"].shape[1]
    in_tokens = enc["attention_mask"].sum(dim=1).tolist()
    res = []
    for i in range(len(prompts)):
        new = out[i, in_len:]
        res.append({"text": tokenizer.decode(new, skip_special_tokens=True).strip(),
                    "input_tokens": int(in_tokens[i]),
                    "output_tokens": int((new != tokenizer.pad_token_id).sum()),
                    "latency_ms": elapsed_ms / len(prompts)})
    return res


def generate_safe(model_id, prompts, max_new_tokens, **gen_kwargs):
    """При нехватке памяти батч делится пополам."""
    try:
        return generate_batch(model_id, prompts, max_new_tokens, **gen_kwargs)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(prompts) == 1:
            raise
        mid = len(prompts) // 2
        return (generate_safe(model_id, prompts[:mid], max_new_tokens, **gen_kwargs)
                + generate_safe(model_id, prompts[mid:], max_new_tokens, **gen_kwargs))


# оценка по логитам, вероятности заданных первых токенов (A/B для судей, Yes/No для покрытия)
def _option_ids(tokenizer, opt):
    ids = set()
    for s in (opt, " " + opt):
        t = tokenizer.encode(s, add_special_tokens=False)
        if len(t) == 1:
            ids.add(t[0])
    if not ids:
        raise ValueError(f"Токен для варианта {opt!r} не найден в токенизаторе")
    return sorted(ids)


# Логарифмы вероятностей каждого варианта (первый токен ответа после chat-шаблона) 
# без генерации текста
# Возвращает список dict(logp=[...], mass=P(любой из вариантов), input_tokens, latency_ms)
@torch.inference_mode()
def score_options(model_id, prompts, options):
    tokenizer, model = load_local_model(model_id)
    texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}],
                                           tokenize=False, add_generation_prompt=True) for p in prompts]
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    t0 = time.perf_counter()
    out = model.generate(**enc, max_new_tokens=1, do_sample=False, output_scores=True,
                         return_dict_in_generate=True, pad_token_id=tokenizer.pad_token_id)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    logp = torch.log_softmax(out.scores[0].float(), dim=-1)                       # [B, V]
    per_opt = torch.stack([torch.logsumexp(logp[:, _option_ids(tokenizer, o)], dim=-1) for o in options], dim=1)
    mass = torch.logsumexp(per_opt, dim=1).exp()
    in_tokens = enc["attention_mask"].sum(dim=1).tolist()
    return [{"logp": per_opt[i].tolist(), "mass": float(mass[i]), "input_tokens": int(in_tokens[i]),
             "latency_ms": elapsed_ms / len(prompts)} for i in range(len(prompts))]


def score_safe(model_id, prompts, options):
    try:
        return score_options(model_id, prompts, options)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(prompts) == 1:
            raise
        mid = len(prompts) // 2
        return score_safe(model_id, prompts[:mid], options) + score_safe(model_id, prompts[mid:], options)


In [ ]:
# ============================== Вспомогательные функции ==============================
def n_words(text): return len(text.split())
def n_sentences(text): return len([s for s in re.split(r"(?<=[.!?])\s+|\n+", text) if len(s.split()) >= 3])
def code_fraction(text):
    code = sum(len(m) for m in re.findall(r"```.*?```", text, flags=re.S))
    return code / max(len(text), 1)
def ascii_ratio(text): return sum(c.isascii() for c in text) / max(len(text), 1)
def stable_int(*parts, mod=2**31 - 1):
    return int(hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest(), 16) % mod
def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

def variant_id(task_id, src, vt, ratio=1.0):
    """T0001_smol_A, T0001_smol_B125, ... src - метка оцениваемой модели, чей ответ A расширяется/сжимается."""
    return f"{task_id}_{src}_A" if vt == "A" else f"{task_id}_{src}_{vt}{round(ratio * 100)}"

def a_id_of(vid):
    t, s = vid.split("_")[:2]
    return f"{t}_{s}_A"

# ============================== CSV вывод и запись ==============================
def read_csv(path, cols):
    df = pd.read_csv(path, dtype=object, keep_default_na=False)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: нет колонок {missing}")
    return df[cols].copy()

def write_csv_atomic(df, path):
    """Запись через временный файл"""
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        df.to_csv(f, index=False)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


# ============================== 1. tasks ==============================
def load_alpaca_eval():
    url = "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json"
    df = pd.read_json(url).reset_index(drop=True)      
    df.insert(0, "benchmark_id", range(len(df)))
    return df                 # benchmark_id, instruction, output, generator, dataset

def select_tasks(df, n_tasks):
    df = df.copy()
    df["words"] = df["output"].map(n_words)
    df["sents"] = df["output"].map(n_sentences)
    df["code_frac"] = df["output"].map(code_fraction)
    df["ascii"] = (df["instruction"] + df["output"]).map(ascii_ratio)
    ok = df[(df.words.between(MIN_WORDS, MAX_WORDS)) & (df.sents >= MIN_SENTENCES)
            & (df.code_frac <= MAX_CODE_FRACTION) & (df.ascii >= MIN_ASCII_RATIO)].copy()
    ok = ok.drop_duplicates("instruction")
    if len(ok) < 200:
        raise RuntimeError(f"После фильтрации осталось {len(ok)} задач (<200): ослабьте критерии.")
    n_tasks = min(n_tasks, len(ok))
    shares = ok["dataset"].value_counts(normalize=True)          # стратификация по подмножествам
    parts = []
    for cat, share in shares.items():
        sub = ok[ok.dataset == cat]
        parts.append(sub.sample(n=min(max(1, round(share * n_tasks)), len(sub)), random_state=SEED))
    sel = pd.concat(parts)
    if len(sel) > n_tasks:
        sel = sel.sample(n=n_tasks, random_state=SEED)
    elif len(sel) < n_tasks:
        sel = pd.concat([sel, ok.drop(sel.index).sample(n=n_tasks - len(sel), random_state=SEED)])
    sel = sel.sort_values("benchmark_id").reset_index(drop=True)
    rows = []
    for i, r in sel.iterrows():
        rows.append({
            "task_id": f"T{i:04d}", "source_benchmark": "AlpacaEval 2.0",
            "benchmark_id": int(r.benchmark_id), "instruction": r.instruction,
            "answers": "[]",                       # заполняется после Stage 1 (ответ оцениваемой модели)
            "gold_answer": r.output,               # референс AlpacaEval (GPT-4 Turbo), не строгий gold
            "category": r.dataset,
            "selection_reason": (f"reference {r.words} words, {r.sents} sentences, "
                                 f"code_frac={r.code_frac:.2f}: enough content for 50-75% compression "
                                 f"and 125-200% expansion; stratified by subset '{r.dataset}'")})
    return pd.DataFrame(rows, columns=TASK_COLS)


# ============================== 2. variants ==============================
def ensure_a_rows(variants, tasks):
    """Строки A для всех задач; каждая модель отвечает на каждую задачу."""
    have = set(variants.variant_id)
    new = []
    '''
    for i, t in enumerate(tasks.itertuples()):
        vid = variant_id(t.task_id, "A")
        if vid not in have:
            new.append({"variant_id": vid, "task_id": t.task_id, "variant_type": "A", "text": "",
                        "target_length_ratio": 1.0, "actual_length_ratio": "",
                        #"generator_model": EVALUATED_MODELS[i % len(EVALUATED_MODELS)],
                        "generator_model": EVALUATED_MODELS[i],
                        "prompt": t.instruction})
    '''
    for model in EVALUATED_MODELS:
        for t in tasks.itertuples():
            vid = variant_id(t.task_id, SOURCE_TAGS[model], "A")
            if vid not in have:
                new.append({"variant_id": vid, "task_id": t.task_id, "variant_type": "A", "text": "",
                            "target_length_ratio": 1.0, "actual_length_ratio": "",
                            "generator_model": model, "prompt": t.instruction})
    if new:
        variants = pd.concat([variants, pd.DataFrame(new, columns=VARIANT_COLS)], ignore_index=True)
    return variants

# ============================== 3. comparisons ==============================
def build_comparisons(task_ids):
    rows = []
    def add(task_id, src, exp, v1, v2):
        for judge in JUDGES:
            for rep in range(REPEATS):
                left, right = (v1, v2) if rep % 2 == 0 else (v2, v1)
                cid = f"{task_id}_{src}_{exp}_{v1.split('_')[2]}_{v2.split('_')[2]}_{judge}_r{rep}"
                rows.append({"comparison_id": cid, "task_id": task_id, "experiment_type": exp,
                             "variant_left_id": left, "variant_right_id": right,
                             "judge_id": judge, "repeat_index": rep,
                             "random_seed": stable_int(cid, SEED)})
    for tid in task_ids:
        for model in EVALUATED_MODELS:                 # сравнения только внутри одного источника A
            src = SOURCE_TAGS[model]
            V = lambda vt, r=1.0: variant_id(tid, src, vt, r)
            for r in EXPAND_LEVELS:
                add(tid, src, "exp1_B_vs_C", V("B", r), V("C", r))
                add(tid, src, "exp2_C_vs_A", V("C", r), V("A"))
            for r in COMPRESS_LEVELS:
                add(tid, src, "exp3_D_vs_A", V("D", r), V("A"))
            for r in LOSSY_LEVELS:
                add(tid, src, "exp4_E_vs_A", V("E", r), V("A"))
            add(tid, src, "exp5_P_vs_A", V("P"), V("A"))
    return pd.DataFrame(rows, columns=COMPARISON_COLS)


In [ ]:
# =============== Init: файлы создаются, если их нет; если есть - загрузка и продолжение ===============
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Если в папке ноутбука данных нет, 
# нр результаты прошлого запуска подключены как Kaggle Input - идет копирование
def restore_from_previous_run():
    if TASKS_CSV.exists():
        return
    root = Path("/kaggle/input")
    if not root.exists():
        return
    names = ["tasks.csv", "variants.csv", "comparisons.csv", "judgments.csv"]
    for p in root.rglob("tasks.csv"):
        if all((p.parent / n).exists() for n in names):
            for n in names + ["answer_attempts.csv", "claims.csv", "variant_attempts.csv"]:
                if (p.parent / n).exists():
                    shutil.copy(p.parent / n, DATA_DIR / n)
            print(f"[restore] данные восстановлены из {p.parent}")
            return

restore_from_previous_run()

# tasks
if TASKS_CSV.exists():
    tasks = read_csv(TASKS_CSV, TASK_COLS); print(f"[tasks] загружен: {len(tasks)} задач")
else:
    tasks = select_tasks(load_alpaca_eval(), N_TASKS)
    write_csv_atomic(tasks, TASKS_CSV); print(f"[tasks] создан: {len(tasks)} задач")

# variants (строки A)
if VARIANTS_CSV.exists():
    variants = read_csv(VARIANTS_CSV, VARIANT_COLS); print(f"[variants] загружен: {len(variants)} строк")
else:
    variants = pd.DataFrame(columns=VARIANT_COLS, dtype=object); print("[variants] создан")
variants = ensure_a_rows(variants, tasks)
write_csv_atomic(variants, VARIANTS_CSV)

# comparisons (достраивание недостающих id)
plan = build_comparisons(tasks.task_id)
if COMPARISONS_CSV.exists():
    comparisons = read_csv(COMPARISONS_CSV, COMPARISON_COLS)
    comparisons = pd.concat([comparisons, plan[~plan.comparison_id.isin(set(comparisons.comparison_id))]],
                            ignore_index=True)
    print(f"[comparisons] загружен: {len(comparisons)} строк")
else:
    comparisons = plan; print(f"[comparisons] создан: {len(comparisons)} строк")
write_csv_atomic(comparisons, COMPARISONS_CSV)

# judgments
if JUDGMENTS_CSV.exists():
    judgments = read_csv(JUDGMENTS_CSV, JUDGMENT_COLS); print(f"[judgments] загружен: {len(judgments)} строк")
else:
    judgments = pd.DataFrame(columns=JUDGMENT_COLS, dtype=object)
    write_csv_atomic(judgments, JUDGMENTS_CSV); print("[judgments] создан (пустой)")

# журнал попыток генерации ответов A
if ATTEMPTS_CSV.exists():
    attempts = read_csv(ATTEMPTS_CSV, ATTEMPT_COLS); print(f"[attempts] загружен: {len(attempts)} строк")
else:
    attempts = pd.DataFrame(columns=ATTEMPT_COLS, dtype=object)
    write_csv_atomic(attempts, ATTEMPTS_CSV); print("[attempts] создан (пустой)")

# атомарные утверждения ответов A
if CLAIMS_CSV.exists():
    claims = read_csv(CLAIMS_CSV, CLAIM_COLS); print(f"[claims] загружен: {len(claims)} строк")
else:
    claims = pd.DataFrame(columns=CLAIM_COLS, dtype=object)
    write_csv_atomic(claims, CLAIMS_CSV); print("[claims] создан (пустой)")

# журнал попыток вариантов B-P
if VATTEMPTS_CSV.exists():
    vattempts = read_csv(VATTEMPTS_CSV, VATTEMPT_COLS); print(f"[variant_attempts] загружен: {len(vattempts)} строк")
else:
    vattempts = pd.DataFrame(columns=VATTEMPT_COLS, dtype=object)
    write_csv_atomic(vattempts, VATTEMPTS_CSV); print("[variant_attempts] создан (пустой)")


In [ ]:
class BudgetExceeded(Exception):
    pass

def check_budget():
    if time.time() - _T0 > TIME_BUDGET_HOURS * 3600:
        raise BudgetExceeded

def active_task_ids():
    ids = tasks.task_id.tolist()
    return set(ids[:ACTIVE_TASK_LIMIT]) if ACTIVE_TASK_LIMIT else set(ids)

def sync_task_answers():
    """Переносит ответы A оцениваемых моделей в tasks.answers."""
    global tasks
    a = variants[(variants.variant_type == "A") & (variants.text != "")]
    ans = {}
    for r in a.itertuples():
        ans.setdefault(r.task_id, []).append({"generator": r.generator_model, "output": r.text})
    tasks["answers"] = [json.dumps(ans[t], ensure_ascii=False) if t in ans else old
                        for t, old in zip(tasks.task_id, tasks.answers)]
    write_csv_atomic(tasks, TASKS_CSV)


# ---------- генерация (ответы A и варианты B/C/D/E/P) ----------
def run_generation(model_id, unload=True):
    # Дозаполняет пустые ответы A, закреплённые за model_id, после каждого батча сохранение
    global variants
    check_budget()
    mask = ((variants.variant_type == "A") & (variants.generator_model == model_id) & (variants.text == "")
            & (variants.prompt != "") & variants.task_id.isin(active_task_ids()))
    idx = variants.index[mask].tolist()
    if not idx:
        print(f"[skip] {model_id}: нечего генерировать"); return
    a_words = {r.variant_id: n_words(r.text) for r in variants[variants.variant_type == "A"].itertuples() if r.text}

    def max_new(i):
        row = variants.loc[i]
        if row.variant_type == "A":
            return A_MAX_NEW_TOKENS
        words = round(a_words[a_id_of(row.variant_id)] * float(row.target_length_ratio))
        return min(int(words * 1.5) + 60, 1800)

    idx.sort(key=lambda i: -len(variants.at[i, "prompt"]))     # похожие по длине - в один батч
    print(f"[run] {model_id}: осталось {len(idx)}")
    bar = tqdm(total=len(idx), desc=model_id.split("/")[-1])
    try:
        for chunk in chunks(idx, BATCH_GEN):
            check_budget()
            res = generate_safe(model_id, [variants.at[i, "prompt"] for i in chunk],
                                max(max_new(i) for i in chunk))
            new_log = []
            for i, r in zip(chunk, res):
                if not r["text"]:
                    continue                             # пустой вывод - повторится при следующем запуске
                variants.at[i, "text"] = r["text"]
                if variants.at[i, "variant_type"] == "A":
                    variants.at[i, "actual_length_ratio"] = 1.0
                    new_log.append(_attempt_row(variants.at[i, "variant_id"], 1, r, "", True))
                else:
                    variants.at[i, "actual_length_ratio"] = round(
                        n_words(r["text"]) / a_words[a_id_of(variants.at[i, "variant_id"])], 3)
            write_csv_atomic(variants, VARIANTS_CSV)
            if new_log:
                log_attempts(new_log)
            bar.update(len(chunk))
    finally:
        bar.close()
        if unload:
            unload_model(model_id)
        sync_task_answers()


# Контроль длины ответов A; журнал попыток и перегенерация
import itertools
def answer_status(n_w, truncated=False):
    if truncated:
        return "truncated"
    if n_w > A_MAX_WORDS:
        return "too_long"
    if n_w < A_MIN_WORDS:
        return "too_short"
    return "ok"

def badness(n_w, truncated=False):
    """Насколько ответ вне допустимого диапазона (0 - в диапазоне); 
    обрезанный ответ штрафуется."""
    return max(A_MIN_WORDS - n_w, n_w - A_MAX_WORDS, 0) + (1000 if truncated else 0)

def _attempt_row(vid, attempt, res, hint, kept):
    n_w = n_words(res["text"])
    trunc = int(res["output_tokens"] >= A_MAX_NEW_TOKENS)       # упёрлись в лимит токенов -> ответ обрезан
    return {"variant_id": vid, "attempt": attempt, "n_words": n_w, "n_tokens": res["output_tokens"],
            "truncated": trunc, "status": answer_status(n_w, bool(trunc)), "kept": int(kept), "hint": hint}

def log_attempts(rows):
    global attempts
    attempts = pd.concat([attempts, pd.DataFrame(rows, columns=ATTEMPT_COLS)], ignore_index=True)
    write_csv_atomic(attempts, ATTEMPTS_CSV)

def _attempt_state():
    """{variant_id: (число попыток, обрезан ли сохранённый ответ)}; 
    нет записи в журнале = 1 попытка."""
    if attempts.empty:
        return {}
    a = attempts.copy()
    for c in ("attempt", "truncated", "kept"):
        a[c] = pd.to_numeric(a[c])
    state = {}
    for vid, g in a.groupby("variant_id"):
        kept = g[g.kept == 1]
        state[vid] = (int(g.attempt.max()), bool(kept.truncated.iloc[-1]) if len(kept) else False)
    return state

def answers_needing_regen(model_id):
    """Ответы A модели вне допустимой длины, у которых остались попытки."""
    state = _attempt_state()
    has_derived = {a_id_of(v) for v in variants.variant_id[variants.variant_type != "A"]}
    a = variants[(variants.variant_type == "A") & (variants.generator_model == model_id)
                 & (variants.text != "") & variants.task_id.isin(active_task_ids())]
    todo, blocked = [], 0
    for i, r in a.iterrows():
        used, trunc = state.get(r.variant_id, (1, False))
        status = answer_status(n_words(r.text), trunc)
        if status == "ok" or used >= 1 + A_MAX_REGEN:
            continue
        if r.variant_id in has_derived:
            blocked += 1
            continue
        todo.append({"idx": i, "vid": r.variant_id, "task_id": r.task_id, "status": status,
                     "attempt": used + 1, "old_bad": badness(n_words(r.text), trunc)})
    if blocked:
        print(f"[regen] пропущено {blocked}: от этих ответов уже построены B-E/P (см. reset_answers)")
    return todo

def regen_prompt(instruction, status):
    if status == "too_short":
        hint = f"Please give a detailed and complete answer of about {A_TARGET_WORDS} words."
    else:                                            # too_long / truncated
        hint = f"Please answer concisely, in about {A_TARGET_WORDS} words."
    return f"{instruction}\n\n{hint}", hint

def regenerate_answers(model_id):
    """Перегенирирует слишком длинные/короткие/обрезанные ответы A (модель остаётся загруженной!).
    Попытка 2: жадная генерация + подсказка о длине; попытки 3+: сэмплирование (T=0.7) + подсказка.
    Новый ответ заменяет старый, только если ближе к допустимому диапазону.
    Использованный промпт записывается вvariants.prompt."""
    instr = dict(zip(tasks.task_id, tasks.instruction))
    for _ in range(A_MAX_REGEN):
        todo = answers_needing_regen(model_id)
        if not todo:
            return
        print(f"[regen] {model_id}: {len(todo)} ответов вне [{A_MIN_WORDS}, {A_MAX_WORDS}] слов или обрезаны")
        todo.sort(key=lambda t: t["attempt"])
        for attempt_no, grp in itertools.groupby(todo, key=lambda t: t["attempt"]):
            kw = {} if attempt_no == 2 else {"do_sample": True, "temperature": 0.7, "top_p": 0.9,
                                              "seed": SEED + attempt_no}
            for chunk in chunks(list(grp), BATCH_GEN):
                check_budget()
                built = [regen_prompt(instr[t["task_id"]], t["status"]) for t in chunk]
                res = generate_safe(model_id, [b[0] for b in built], A_MAX_NEW_TOKENS, **kw)
                log = []
                for t, (prompt, hint), r in zip(chunk, built, res):
                    new_bad = (badness(n_words(r["text"]), r["output_tokens"] >= A_MAX_NEW_TOKENS)
                               if r["text"] else float("inf"))
                    keep_new = new_bad < t["old_bad"]
                    if keep_new:
                        variants.at[t["idx"], "text"] = r["text"]
                        variants.at[t["idx"], "prompt"] = prompt
                    log.append(_attempt_row(t["vid"], attempt_no, r, hint, keep_new))
                write_csv_atomic(variants, VARIANTS_CSV)
                log_attempts(log)

# Перезапуск
def reset_answers(task_ids, model_id=None):
    """Стирает ответы A выбранных задач (по одной или всем оцениваемым моделям) и всё, что из них построено:
    варианты B-E/P, вердикты судей, журнал попыток. После этого Stage 1 сгенерирует эти ответы заново."""
    global variants, judgments, attempts
    tags = [SOURCE_TAGS[m] for m in ([model_id] if model_id else EVALUATED_MODELS)]
    prefixes = tuple(f"{t}_{s}_" for t in task_ids for s in tags)
    chain = variants.variant_id.str.startswith(prefixes)
    chain_ids = set(variants.variant_id[chain])
    instr = dict(zip(tasks.task_id, tasks.instruction))
    comp_ids = set(comparisons.comparison_id[comparisons.variant_left_id.isin(chain_ids)
                                             | comparisons.variant_right_id.isin(chain_ids)])
    variants = variants[~(chain & (variants.variant_type != "A"))].reset_index(drop=True)
    is_a = variants.variant_id.isin(chain_ids) & (variants.variant_type == "A")
    variants.loc[is_a, ["text", "actual_length_ratio"]] = ""
    variants.loc[is_a, "prompt"] = variants.loc[is_a, "task_id"].map(instr)
    judgments = judgments[~judgments.comparison_id.isin(comp_ids)].reset_index(drop=True)
    attempts = attempts[~attempts.variant_id.isin(chain_ids)].reset_index(drop=True)
    for df, path in ((variants, VARIANTS_CSV), (judgments, JUDGMENTS_CSV), (attempts, ATTEMPTS_CSV)):
        write_csv_atomic(df, path)
    sync_task_answers()

def show_answer_lengths():
    a = variants[(variants.variant_type == "A") & (variants.text != "")].copy()
    if a.empty:
        return
    state = _attempt_state()
    a["words"] = a.text.map(n_words)
    a["attempts"] = [state.get(v, (1, False))[0] for v in a.variant_id]
    a["status"] = [answer_status(w, state.get(v, (1, False))[1]) for w, v in zip(a.words, a.variant_id)]
    print(f"\nОтветы A (допустимо {A_MIN_WORDS}-{A_MAX_WORDS} слов):")
    print(a.groupby("generator_model").agg(n=("words", "size"), median_words=("words", "median"),
                                           regenerated=("attempts", lambda x: int((x > 1).sum()))).to_string())
    print(a.groupby(["generator_model", "status"]).size().unstack(fill_value=0).to_string())


def stage_answers():
    for m in EVALUATED_MODELS:                 # модели строго по очереди
        try:
            run_generation(m, unload=False)    # 1) первичная генерация (естественные ответы)
            regenerate_answers(m)              # 2) перегенерация слишком длинных/коротких/обрезанных
        finally:
            unload_model(m)
            sync_task_answers()


In [ ]:
# ============================== Stage 2: варианты B, C, D, E, P ==============================
import random, math, itertools

# ---------- очистка вывода модели ----------
PRE_RE = re.compile(r"^\s*(?:(?:sure|certainly|of course|okay|ok|absolutely)\b|"
                    r"(?:here(?:'s| is| are)|below is|this is (?:the|a|my)|the following is)\b)", re.I)
TAIL_RE = re.compile(r"^\W*(?:note\b|word count|\(?\d+ words\)?|this (?:version|rewrite|text|answer|response)\b|"
                     r"i (?:have|'ve|hope|kept|added|removed)\b|let me know|hope this)", re.I)

def has_preamble(text):
    first = text.strip().split("\n", 1)[0]
    return bool(PRE_RE.match(first)) and len(first) < 160

def clean_output(text, strip_preamble=True):
    """Убирает вступление "Here is the rewritten answer", хвоствы, счётчик слов, кавычки."""
    t = text.strip()
    if strip_preamble:
        for _ in range(2):
            first, _sep, rest = t.partition("\n")
            if has_preamble(first) and rest.strip():
                t = rest.strip()
            else:
                break
    paras = t.split("\n\n")
    while len(paras) > 1 and TAIL_RE.match(paras[-1].strip()):
        paras.pop()
    t = "\n\n".join(paras).strip()
    t = re.sub(r"\s*\(\s*(?:word count|words)\s*[:=]?\s*\d+[^)]*\)\s*$", "", t, flags=re.I).strip()
    if len(t) > 2 and t[0] in "\"“" and t[-1] in "\"”":
        t = t[1:-1].strip()
    return t

def src_of(vid):
    return vid.split("_")[1]


# ---------- атомарные утверждения ответа A (для D и E) ----------
def parse_claims(text):
    out, seen = [], set()
    for line in text.splitlines():
        m = re.match(r"^\s*(?:\d+[.)]|[-*•])\s*(.+?)\s*$", line)
        if m:
            c = m.group(1).strip().strip("*").strip()
            if len(c.split()) >= 3 and c.lower() not in seen:
                seen.add(c.lower()); out.append(c)
    return out[:30]

def choose_removed(a_id, n):
    """Какие утверждения удаляются в E (детерминированно по a_id); при <3 утверждений-пусто."""
    if n < 3:
        return []
    k = max(1, round(E_REMOVE_FRAC * n))
    return sorted(random.Random(stable_int(a_id, SEED)).sample(range(n), k))

def _claims_dict():
    return {r.a_variant_id: (json.loads(r.claims_json), json.loads(r.removed_json)) for r in claims.itertuples()}

def extract_claims(model_id):
    """Извлекает утверждения из каждого ответа A (модель model_id). Результат пишется в claims.csv."""
    global claims
    check_budget()
    a = variants[(variants.variant_type == "A") & (variants.text != "") & variants.task_id.isin(active_task_ids())]
    done = set(claims.a_variant_id)
    todo = [r for r in a.itertuples() if r.variant_id not in done]
    if not todo:
        print("[skip] claims: нечего извлекать"); return
    instr = dict(zip(tasks.task_id, tasks.instruction))
    todo.sort(key=lambda r: -len(r.text))
    print(f"[run] claims ({model_id}): осталось {len(todo)}")
    bar = tqdm(total=len(todo), desc="claims")
    for chunk in chunks(todo, BATCH_GEN):
        check_budget()
        prompts = [CLAIMS_PROMPT.format(instruction=instr[r.task_id], answer=r.text) for r in chunk]
        res = generate_safe(model_id, prompts, CLAIMS_MAX_NEW_TOKENS)
        parsed = [parse_claims(x["text"]) for x in res]
        weak = [i for i, p in enumerate(parsed) if len(p) < MIN_CLAIMS]
        if weak:                                           # повтор со сэмплированием, берём лучший результат
            res2 = generate_safe(model_id, [prompts[i] for i in weak], CLAIMS_MAX_NEW_TOKENS,
                                 do_sample=True, temperature=0.7, top_p=0.9, seed=SEED)
            for i, x in zip(weak, res2):
                p2 = parse_claims(x["text"])
                if len(p2) > len(parsed[i]):
                    parsed[i] = p2
        rows = [{"a_variant_id": r.variant_id, "claims_json": json.dumps(p, ensure_ascii=False),
                 "removed_json": json.dumps(choose_removed(r.variant_id, len(p)))} for r, p in zip(chunk, parsed)]
        claims = pd.concat([claims, pd.DataFrame(rows, columns=CLAIM_COLS)], ignore_index=True)
        write_csv_atomic(claims, CLAIMS_CSV)
        bar.update(len(chunk))
    bar.close()


# ---------- журнал попыток вариантов ----------
def log_vattempts(rows):
    global vattempts
    vattempts = pd.concat([vattempts, pd.DataFrame(rows, columns=VATTEMPT_COLS)], ignore_index=True)
    write_csv_atomic(vattempts, VATTEMPTS_CSV)

def _vstate():
    # {variant_id: dict(n=число попыток, ok=принят ли сохранённый текст, score=его штраф, 
    # feedback=замечания к нему)}
    if vattempts.empty:
        return {}
    a = vattempts.copy()
    for c in ("attempt", "ok", "score", "kept"):
        a[c] = pd.to_numeric(a[c])
    state = {}
    for vid, g in a.groupby("variant_id"):
        kept = g[g.kept == 1]
        last = kept.iloc[-1] if len(kept) else None
        state[vid] = {"n": int(g.attempt.max()),
                      "ok": bool(last.ok) if last is not None else False,
                      "score": float(last.score) if last is not None else float("inf"),
                      "feedback": str(last.feedback) if last is not None else ""}
    return state

def variant_done_ids():
    """Варианты с текстом, которые приняты или исчерпали попытки (от них можно строить C)."""
    state = _vstate()
    has_text = set(variants.variant_id[variants.text != ""])
    return {v for v in has_text if v in state and (state[v]["ok"] or state[v]["score"] <= RETRY_SCORE
                                                   or state[v]["n"] >= VARIANT_MAX_TRIES)}


# ---------- строки вариантов и промпты ----------
def build_prompt(vt, instruction, a_text, words, cl):
    lo, hi = round(words * (1 - LEN_TOL)), round(words * (1 + LEN_TOL))
    claims_list, removed = cl if cl else ([], [])
    claims_block = ("\nClaims that must all remain in the text:\n" + "\n".join(f"- {c}" for c in claims_list)
                    ) if claims_list else ""
    removed_block = ("Points to delete:\n" + "\n".join(f"- {claims_list[i]}" for i in removed)
                     ) if removed else "Points to delete: about 30% of the key points (choose the least central ones)."
    return PROMPTS[vt].format(instruction=instruction, answer=a_text, words=words, lo=lo, hi=hi,
                              claims_block=claims_block, removed_block=removed_block)

def ensure_derived_rows(variants, tasks):
    """Создаёт строки B/P (сразу), D/E (после извлечения утверждений) и C (после готового B того же уровня)."""
    have = set(variants.variant_id)
    a = variants[(variants.variant_type == "A") & (variants.text != "")]
    instr = dict(zip(tasks.task_id, tasks.instruction))
    text_of = dict(zip(variants.variant_id, variants.text))
    cl_d, done = _claims_dict(), variant_done_ids()
    new = []
    for r in a.itertuples():
        src, a_w = SOURCE_TAGS[r.generator_model], n_words(r.text)
        for vt, ratio in DERIVED_PLAN:
            vid = variant_id(r.task_id, src, vt, ratio)
            if vid in have:
                continue
            if vt in ("D", "E") and r.variant_id not in cl_d:
                continue                                          # ждём извлечения утверждений
            words = round(a_w * ratio)
            if vt == "C":
                b_id = variant_id(r.task_id, src, "B", ratio)
                if b_id not in done:
                    continue                                      # ждём готового B
                words = n_words(text_of[b_id])                    # длина C = фактическая длина B
            prompt = build_prompt(vt, instr[r.task_id], r.text, words, cl_d.get(r.variant_id))
            new.append({"variant_id": vid, "task_id": r.task_id, "variant_type": vt, "text": "",
                        "target_length_ratio": ratio, "actual_length_ratio": "",
                        "generator_model": REWRITER_MODEL if vt == "P" else GENERATOR_MODEL, "prompt": prompt})
    if new:
        variants = pd.concat([variants, pd.DataFrame(new, columns=VARIANT_COLS)], ignore_index=True)
    return variants

def refresh_variant_rows():
    global variants
    variants = ensure_derived_rows(variants, tasks)
    write_csv_atomic(variants, VARIANTS_CSV)


# ---------- оценка варианта ----------
def evaluate_variant(t, text, tokens, chunk_max_new, covered):
    """Проверки: длина, обрезание, вступление, покрытие утверждений (D/E). Возвращает метрики, штраф score и текст замечаний."""
    vt, W, a_w = t["vt"], t["W"], t["a_words"]
    n_w = n_words(text)
    if n_w == 0:
        return {"n_words": 0, "rel": 0.0, "truncated": 0, "preamble": 0, "coverage": "", "removed_present": "",
                "ok": 0, "score": 99.0, "feedback": "The text was empty."}
    problems, score = [], 0.0
    if vt == "E":
        rel = n_w / a_w
        err = max(E_LEN_BAND[0] - rel, rel - E_LEN_BAND[1], 0)
        if err > 0:
            problems.append(f"The text has {n_w} words ({rel:.0%} of the original); it must be between "
                            f"{E_LEN_BAND[0]:.0%} and {E_LEN_BAND[1]:.0%} of the original length.")
    else:
        rel = n_w / W
        err = max(abs(rel - 1) - LEN_TOL, 0)
        if err > 0:
            problems.append(f"The text has {n_w} words, which is too {'long' if rel > 1 else 'short'}; it must be about "
                            f"{W} words (between {round(W * (1 - LEN_TOL))} and {round(W * (1 + LEN_TOL))}).")
    score += err
    trunc = int(tokens >= chunk_max_new)
    if trunc:
        score += 1.0
        problems.append("The text was cut off; make it shorter and finish it properly.")
    pre = int(has_preamble(text) and not t["a_has_pre"])
    if pre:
        score += 1.0
        problems.append("The text must start directly with the answer: no introduction, notes or word count.")
    coverage = removed_present = ""
    cl = t["claims"]
    if covered is not None and cl and len(cl[0]) >= 3:
        claims_list, removed = cl
        flags = [bool(covered.get(k, False)) for k in range(len(claims_list))]
        if vt == "D":
            coverage = sum(flags) / len(flags)
            need = D_MIN_COVERAGE.get(t["ratio"], 0.9)
            if coverage < need:
                score += need - coverage
                miss = [c for c, f in zip(claims_list, flags) if not f][:8]
                problems.append("The text omits these essential claims, include them: " + "; ".join(miss))
        elif vt == "E":
            kept_idx = [k for k in range(len(claims_list)) if k not in removed]
            coverage = sum(flags[k] for k in kept_idx) / len(kept_idx) if kept_idx else 1.0
            removed_present = sum(flags[k] for k in removed) / len(removed) if removed else 0.0
            if coverage < E_MIN_KEPT:
                score += E_MIN_KEPT - coverage
                miss = [claims_list[k] for k in kept_idx if not flags[k]][:8]
                problems.append("These points must stay in the text but are missing: " + "; ".join(miss))
            if removed_present > E_MAX_REMOVED:
                score += removed_present - E_MAX_REMOVED
                bad = [claims_list[k] for k in removed if flags[k]][:8]
                problems.append("These points must be deleted completely but are still present: " + "; ".join(bad))
    return {"n_words": n_w, "rel": round(rel, 3), "truncated": trunc, "preamble": pre,
            "coverage": "" if coverage == "" else round(coverage, 3),
            "removed_present": "" if removed_present == "" else round(removed_present, 3),
            "ok": int(not problems), "score": round(score, 4), "feedback": " ".join(problems)}

def retry_prompt(base, kept_text, feedback):
    return (f"{base}\n\n[PREVIOUS ATTEMPT]\n{kept_text}\n[END OF PREVIOUS ATTEMPT]\n"
            f"Problems with the previous attempt: {feedback}\n"
            f"Write a corrected version that fixes these problems. {OUTPUT_RULES}")

def variants_needing_work(model_id, types):
    state = _vstate()
    a_rows = variants[variants.variant_type == "A"]
    a_text = dict(zip(a_rows.variant_id, a_rows.text))
    text_of = dict(zip(variants.variant_id, variants.text))
    cl_d = _claims_dict()
    sel = variants[variants.variant_type.isin(types) & (variants.generator_model == model_id)
                   & (variants.prompt != "") & variants.task_id.isin(active_task_ids())]
    todo = []
    for i, r in sel.iterrows():
        st = state.get(r.variant_id)
        n = st["n"] if st else 0
        if n >= VARIANT_MAX_TRIES or (st and r.text and (st["ok"] or st["score"] <= RETRY_SCORE)):
            continue
        a_id = a_id_of(r.variant_id)
        if not a_text.get(a_id):
            continue
        a_w, ratio = n_words(a_text[a_id]), float(r.target_length_ratio)
        W = round(a_w * ratio)
        if r.variant_type == "C":
            W = n_words(text_of.get(variant_id(r.task_id, src_of(r.variant_id), "B", ratio), "")) or W
        todo.append({"idx": i, "vid": r.variant_id, "vt": r.variant_type, "ratio": ratio, "a_id": a_id,
                     "a_words": a_w, "a_has_pre": has_preamble(a_text[a_id]), "W": W, "base": r.prompt,
                     "attempt": n + 1, "kept_text": r.text if st else "", "feedback": st["feedback"] if st else "",
                     "old_score": st["score"] if st else None, "claims": cl_d.get(a_id)})
    return todo

def _process_chunk(model_id, chunk, res, attempt_no, chunk_max_new):
    cleaned = [clean_output(r["text"], strip_preamble=not t["a_has_pre"]) for t, r in zip(chunk, res)]
    cov_prompts, cov_idx = [], []                              # покрытие утверждений для D/E
    for j, (t, txt) in enumerate(zip(chunk, cleaned)):
        if t["vt"] in ("D", "E") and txt and t["claims"] and len(t["claims"][0]) >= 3:
            for k, c in enumerate(t["claims"][0]):
                cov_prompts.append(COVER_PROMPT.format(text=txt, claim=c)); cov_idx.append((j, k))
    covered = {}
    if cov_prompts:
        scored = []
        for sub in chunks(cov_prompts, BATCH_SCORE):
            scored += score_safe(model_id, sub, ("Yes", "No"))
        for (j, k), s in zip(cov_idx, scored):
            covered.setdefault(j, {})[k] = s["logp"][0] > s["logp"][1]
    log = []
    for j, (t, r, txt) in enumerate(zip(chunk, res, cleaned)):
        ev = evaluate_variant(t, txt, r["output_tokens"], chunk_max_new, covered.get(j))
        keep = t["old_score"] is None or ev["score"] < t["old_score"]
        if keep and txt:
            variants.at[t["idx"], "text"] = txt
            variants.at[t["idx"], "actual_length_ratio"] = round(ev["n_words"] / t["a_words"], 3)
        log.append({"variant_id": t["vid"], "attempt": attempt_no, "n_words": ev["n_words"],
                    "target_words": t["W"] if t["vt"] != "E" else round(t["a_words"] * t["ratio"]),
                    "rel": ev["rel"], "truncated": ev["truncated"], "preamble": ev["preamble"],
                    "coverage": ev["coverage"], "removed_present": ev["removed_present"], "ok": ev["ok"],
                    "score": ev["score"], "kept": int(keep and bool(txt)), "feedback": ev["feedback"][:800]})
    return log

def run_variants(model_id, types):
    """Генерация вариантов с контролем качества:до VARIANT_MAX_TRIES попыток; 
    повторная идет с обратной связью, что не так в сохранённом тексте,
    с третей семплирование. Сохраняется лучший вариант по штрафу;
    вариант со штрафом<=RETRY_SCORE повторно не генерируется."""
    global variants
    tok_total, sec_total = 0, 0.0
    for _ in range(VARIANT_MAX_TRIES):
        check_budget()
        todo = variants_needing_work(model_id, types)
        if not todo:
            break
        print(f"[variants] {model_id}: {len(todo)} вариантов к генерации/доработке ({','.join(types)})")
        todo.sort(key=lambda t: t["attempt"])
        for attempt_no, grp in itertools.groupby(todo, key=lambda t: t["attempt"]):
            kw = {} if attempt_no <= 2 else {"do_sample": True, "temperature": 0.7, "top_p": 0.9,
                                              "seed": SEED + attempt_no}
            # похожие по целевой длине - в один батч (иначе батч ждёт самую длинную генерацию)
            grp = sorted(grp, key=lambda t: -(t["a_words"] if t["vt"] == "E" else t["W"]))
            bar = tqdm(total=len(grp), desc=f"attempt {attempt_no}")
            for chunk in chunks(grp, BATCH_VARIANTS):
                check_budget()
                prompts = [t["base"] if (attempt_no == 1 or not t["kept_text"])
                           else retry_prompt(t["base"], t["kept_text"], t["feedback"]) for t in chunk]
                chunk_max_new = max(min(int((t["a_words"] if t["vt"] == "E" else t["W"]) * 1.5) + 80, 1800)
                                    for t in chunk)
                t0 = time.perf_counter()
                res = generate_safe(model_id, prompts, chunk_max_new, **kw)
                sec_total += time.perf_counter() - t0
                tok_total += sum(r["output_tokens"] for r in res)
                log = _process_chunk(model_id, chunk, res, attempt_no, chunk_max_new)
                write_csv_atomic(variants, VARIANTS_CSV)
                log_vattempts(log)
                bar.update(len(chunk))
            bar.close()
    if sec_total:
        print(f"[gen] сгенерировано {tok_total} токенов за {sec_total:.0f} с ({tok_total / sec_total:.0f} ток/с)")

def stage_variants():
    models = list(dict.fromkeys([GENERATOR_MODEL, REWRITER_MODEL]))
    for m in models:                                               # модели строго по очереди
        try:
            if m == GENERATOR_MODEL:
                extract_claims(m)
            refresh_variant_rows()
            run_variants(m, ("B", "D", "E", "P"))
            refresh_variant_rows()                                 # C создаётся по готовым B
            run_variants(m, ("C", "P"))
        finally:
            unload_model(m)

# Нужно после смены промптов/критериев.
def reset_derived(task_ids=None, drop_claims=False):
    """Удаляет варианты B-P выбранных задач (по умолчанию - активных), 
    их вердикты и журнал попыток.
    Ответы A сохраняются."""
    global variants, judgments, vattempts, claims
    ids = set(task_ids) if task_ids else active_task_ids()
    drop = variants.task_id.isin(ids) & (variants.variant_type != "A")
    drop_ids = set(variants.variant_id[drop])
    comp_ids = set(comparisons.comparison_id[comparisons.variant_left_id.isin(drop_ids)
                                             | comparisons.variant_right_id.isin(drop_ids)])
    variants = variants[~drop].reset_index(drop=True)
    judgments = judgments[~judgments.comparison_id.isin(comp_ids)].reset_index(drop=True)
    vattempts = vattempts[~vattempts.variant_id.isin(drop_ids)].reset_index(drop=True)
    pairs = [(variants, VARIANTS_CSV), (judgments, JUDGMENTS_CSV), (vattempts, VATTEMPTS_CSV)]
    if drop_claims:
        claims = claims[~claims.a_variant_id.str.split("_").str[0].isin(ids)].reset_index(drop=True)
        pairs.append((claims, CLAIMS_CSV))
    for df, path in pairs:
        write_csv_atomic(df, path)
    print(f"[reset] удалено вариантов: {len(drop_ids)}, вердиктов: {len(comp_ids)}")

def show_variant_quality():
    d = variants[(variants.variant_type != "A") & (variants.text != "")].copy()
    if d.empty:
        return
    state = _vstate()
    d["ratio"] = d.target_length_ratio.astype(float)
    d["attempts"] = [state.get(v, {"n": 0})["n"] for v in d.variant_id]
    d["ok"] = [state.get(v, {"ok": False})["ok"] for v in d.variant_id]
    d["pre"] = d.text.map(has_preamble)
    a_rows = variants[variants.variant_type == "A"]
    a_len = {v: n_words(t) for v, t in zip(a_rows.variant_id, a_rows.text)}
    d["rel_A"] = [n_words(t) / a_len[a_id_of(v)] for t, v in zip(d.text, d.variant_id)]
    print("\nКачество вариантов B-P:")
    print(d.groupby(["variant_type", "ratio"]).agg(n=("ok", "size"), accepted=("ok", "mean"),
          len_vs_A=("rel_A", "mean"), attempts=("attempts", "mean"), preamble=("pre", "mean")).round(2).to_string())
    text_of = dict(zip(variants.variant_id, variants.text))
    diffs = [abs(n_words(text_of[v.replace("_C", "_B", 1)]) / max(n_words(t), 1) - 1)
             for v, t in zip(d.variant_id, d.text) if v.split("_")[2].startswith("C") and v.replace("_C", "_B", 1) in text_of
             and text_of[v.replace("_C", "_B", 1)]]
    if diffs:
        print(f"\nРазница длин B и C одного уровня: среднее {sum(diffs)/len(diffs):.1%}, максимум {max(diffs):.1%}")
    if len(vattempts):
        a = vattempts.copy()
        for c in ("coverage", "removed_present"):
            a[c] = pd.to_numeric(a[c], errors="coerce")
        a["vt"] = a.variant_id.str.split("_").str[2].str[0]
        print("\nПокрытие утверждений (последняя попытка): D - доля сохранённых; E - сохранённых и ещё присутствующих удалённых")
        print(a.sort_values("attempt").groupby("variant_id").tail(1).groupby("vt")[["coverage", "removed_present"]]
              .mean().loc[lambda x: x.index.isin(["D", "E"])].round(2).to_string())


def usable_variant_ids():
    """Варианты, допущенные к судейству: A всегда; B-P - с текстом и штрафом <= USABLE_SCORE."""
    state = _vstate()
    has_text = set(variants.variant_id[variants.text != ""])
    ids = {v for v in has_text if v.endswith("_A")}
    ids |= {v for v in has_text if v in state and state[v]["score"] <= USABLE_SCORE}
    return ids

# Почему варианты не проходят проверки
def diagnose_variants():
    # Доля провалов по причинам для каждого типа и уровня в первой попытке
    if vattempts.empty:
        print("Журнал попыток пуст"); return
    a = vattempts.copy()
    for c in ("attempt", "rel", "truncated", "preamble", "coverage", "removed_present", "ok", "score", "kept"):
        a[c] = pd.to_numeric(a[c], errors="coerce")
    a["vt"] = a.variant_id.str.split("_").str[2]
    a["type"], a["level"] = a.vt.str[0], a.vt.str[1:]
    first = a[a.attempt == 1].copy()
    first["len_bad"] = [(r < E_LEN_BAND[0] or r > E_LEN_BAND[1]) if t == "E" else abs(r - 1) > LEN_TOL
                        for t, r in zip(first.type, first.rel)]
    need = {"D75": D_MIN_COVERAGE[0.75], "D50": D_MIN_COVERAGE[0.5]}
    first["cov_bad"] = [(c < need.get(v, E_MIN_KEPT)) if (t in ("D", "E") and pd.notna(c)) else False
                        for t, v, c in zip(first.type, first.vt, first.coverage)]
    first["rem_bad"] = first.removed_present > E_MAX_REMOVED
    print("Первая попытка: доля вариантов с проблемой")
    print(first.groupby(["type", "level"]).agg(n=("ok", "size"), passed=("ok", "mean"), length=("len_bad", "mean"),
          cut_off=("truncated", "mean"), intro=("preamble", "mean"), coverage=("cov_bad", "mean"),
          removed_still_there=("rem_bad", "mean"), median_rel=("rel", "median")).round(2).to_string())
    last = a.sort_values("attempt").groupby("variant_id").tail(1)
    kept = a[a.kept == 1].sort_values("attempt").groupby("variant_id").tail(1)
    print("\nИтог после всех попыток: принято (ok) / пригодно к судейству (штраф <= USABLE_SCORE)")
    print(kept.assign(usable=kept.score <= USABLE_SCORE).groupby(["type", "level"])[["ok", "usable"]].mean().round(2).to_string())


In [ ]:
# ============================== Stage 3: судьи (вердикт по логитам токенов A/B) ==============================
def run_judging(judge_label):
    """Вердикт = какой из токенов A/B вероятнее после промпта. Пишется p_left (вероятность «левого» ответа),
    масса токенов A/B (если мала - invalid). Позиционное смещение вычитается позже в calibrate_judgments()."""
    global judgments
    check_budget()
    model_id = JUDGE_MODELS[judge_label]
    text_of = dict(zip(variants.variant_id, variants.text))
    instr = dict(zip(tasks.task_id, tasks.instruction))
    done = set(judgments.comparison_id)
    c = comparisons[(comparisons.judge_id == judge_label) & ~comparisons.comparison_id.isin(done)
                    & comparisons.task_id.isin(active_task_ids())]
    c = c[(c.variant_left_id.map(text_of).fillna("") != "") & (c.variant_right_id.map(text_of).fillna("") != "")]
    if JUDGE_ONLY_USABLE:                                  # судим только пригодные варианты (экономия вызовов)
        n_before = len(c)
        usable = usable_variant_ids()
        c = c[c.variant_left_id.isin(usable) & c.variant_right_id.isin(usable)]
        nw = lambda vid: len(text_of[vid].split())
        is_e1 = c.experiment_type == "exp1_B_vs_C"          # B и C должны совпадать по длине
        len_ok = [abs(nw(l) / nw(r) - 1) <= PAIR_LEN_TOL for l, r in zip(c.variant_left_id, c.variant_right_id)]
        c = c[~is_e1 | pd.Series(len_ok, index=c.index)]
        print(f"[eligible] {judge_label}: к судейству допущено {len(c)} из {n_before} непросуженных сравнений")
    rows = sorted(c.itertuples(), key=lambda r: -(len(text_of[r.variant_left_id]) + len(text_of[r.variant_right_id])))
    if not rows:
        print(f"[skip] {judge_label}: нечего судить"); return
    print(f"[run] {judge_label} ({model_id}): осталось {len(rows)}")
    bar = tqdm(total=len(rows), desc=judge_label)
    try:
        for chunk in chunks(rows, BATCH_JUDGE):
            check_budget()
            prompts = [JUDGE_PROMPT.format(instruction=instr[r.task_id], left=text_of[r.variant_left_id],
                                           right=text_of[r.variant_right_id]) for r in chunk]
            res = score_safe(model_id, prompts, ("A", "B"))
            new = []
            for r, x in zip(chunk, res):
                lp_a, lp_b = x["logp"]
                p_left = 1 / (1 + math.exp(max(min(lp_b - lp_a, 50), -50)))
                winner = "invalid" if x["mass"] < JUDGE_MIN_MASS else ("left" if p_left > 0.5 else "right")
                raw = json.dumps({"lpA": round(lp_a, 4), "lpB": round(lp_b, 4), "p_left": round(p_left, 4),
                                  "mass": round(x["mass"], 4)})
                new.append({"judgment_id": f"J_{r.comparison_id}", "comparison_id": r.comparison_id,
                            "winner": winner, "raw_response": raw, "latency_ms": round(x["latency_ms"], 1),
                            "input_tokens": x["input_tokens"], "output_tokens": 1})
            judgments = pd.concat([judgments, pd.DataFrame(new, columns=JUDGMENT_COLS)], ignore_index=True)
            write_csv_atomic(judgments, JUDGMENTS_CSV)
            bar.update(len(chunk))
    finally:
        bar.close()
        unload_model(model_id)

def stage_judging():
    for label in JUDGES:                       # судьи строго по очереди
        run_judging(label)

def _logit(p):
    p = min(max(p, 1e-4), 1 - 1e-4)
    return math.log(p / (1 - p))

def calibrate_judgments():
    """Вычитает позиционное смещение судьи. 
    Каждая пара показывается в обоих порядках, поэтому среднее logit(p_left) по парам это 
    смещение b судьи (вклад содержания сокращается). 
    Калиброванный вердикт: left, если logit(p_left) > b.
    Исходный вердикт остаётся в judgments.winner для метрик позиционного смещения,   
    калиброванный записывается в raw_response['winner_cal']"""
    global judgments
    if judgments.empty:
        return
    comp = comparisons.set_index("comparison_id")
    info = []
    for cid, raw, w in zip(judgments.comparison_id, judgments.raw_response, judgments.winner):
        try:
            d = json.loads(raw)
        except Exception:
            d = {}
        info.append(d)
    df = pd.DataFrame({"cid": judgments.comparison_id, "winner": judgments.winner,
                       "p_left": [d.get("p_left") for d in info]})
    df["judge"] = df.cid.map(comp.judge_id)
    df["pair"] = ["|".join(sorted([comp.variant_left_id[c], comp.variant_right_id[c]])) for c in df.cid]
    df["left_id"] = df.cid.map(comp.variant_left_id)
    ok = df[df.p_left.notna() & (df.winner != "invalid")].copy()
    ok["logit"] = ok.p_left.astype(float).map(_logit)
    bias = {}
    for judge, g in ok.groupby("judge"):
        both = g.groupby("pair").left_id.transform("nunique") == 2          # пары, показанные в обоих порядках
        base = g[both] if both.any() else g
        bias[judge] = float(base.logit.mean())
    new_raw = []
    for d, raw, judge, w in zip(info, judgments.raw_response, df.judge, judgments.winner):
        if "p_left" in d:
            d["bias"] = round(bias.get(judge, 0.0), 4)
            d["winner_cal"] = ("invalid" if w == "invalid"
                               else ("left" if _logit(float(d["p_left"])) > bias.get(judge, 0.0) else "right"))
            raw = json.dumps(d)
        new_raw.append(raw)
    judgments["raw_response"] = new_raw
    write_csv_atomic(judgments, JUDGMENTS_CSV)
    rows = []
    for judge, g in ok.groupby("judge"):
        sub = judgments[judgments.comparison_id.isin(g.cid)]
        cal = [json.loads(r).get("winner_cal") for r in sub.raw_response]
        rows.append({"judge": judge, "bias_logodds": round(bias[judge], 3), "n": len(g),
                     "P_left_raw": round((sub.winner == "left").mean(), 3),
                     "P_left_calibrated": round(sum(c == "left" for c in cal) / len(cal), 3)})
    print(pd.DataFrame(rows).to_string(index=False))


def safe(stage_fn):
    try:
        stage_fn()
    except BudgetExceeded:
        print("Достигнут лимит времени: всё сохранено. Перезапустите ноутбук и продолжите с этой же ячейки.")

def run_all():
    for fn in (stage_answers, stage_variants, stage_judging):
        safe(fn)
    calibrate_judgments()

def show_status():
    a = variants[variants.variant_type=="A"]
    print("A-ответы:", int((a.text != "").sum()), "/", len(a))
    d = variants[variants.variant_type != "A"]
    if len(d):
        print("\nВарианты (готово/всего):")
        print(d.assign(done=d.text != "").groupby("variant_type").done.agg(["sum", "count"]).to_string())
    print(f"\nУтверждений извлечено для {len(claims)} ответов A")
    print("\nСудейство (готово/всего):")
    done = set(judgments.comparison_id)
    print(comparisons.assign(done=comparisons.comparison_id.isin(done))
          .groupby("judge_id").done.agg(["sum", "count"]).to_string())
    if len(judgments):
        print("\nНевалидных вердиктов:", int((judgments.winner == "invalid").sum()), "из", len(judgments))
        print("Калибровка выполнена:", bool(judgments.raw_response.str.contains("winner_cal").any()))
    show_answer_lengths()
    show_variant_quality()


## Stage 1 - ответы оцениваемых моделей (вариант A)

Сначала каждая модель отвечает на все задачи без подсказок; ответы вне диапазона `A_MIN_WORDS`-`A_MAX_WORDS` слов или обрезанные перегенерируются.


In [ ]:
from huggingface_hub import login
login()

In [ ]:
safe(stage_answers)

## Stage 2 - варианты B, C, D, E, P (с контролем длины, утверждений и вывода)


In [ ]:
safe(stage_variants)

## Stage 3 - судьи


In [ ]:
safe(stage_judging)

Калибровка позиционного смещения (после того, как судьи закончили):


In [ ]:
calibrate_judgments()

## Статус и контроль качества


In [ ]:
show_status()

In [ ]:
diagnose_variants()   # почему варианты не проходят проверки

## Сброс (при смене промптов/критериев)

`reset_derived()` пересоздаёт B-P и вердикты для активных задач, 

`reset_answers([...])` - ответы A.


In [ ]:
# reset_derived()                # Eдалить B-P, их вердикты и журнал попыток (A сохраняются)
# reset_derived(drop_claims=True) # то же+заново извлечь утверждения
# reset_answers(['T0000'])        # заново сгенерировать A для задач
